# 🎬 Montage automatique Locapay

Chargez une **vidéo brute face caméra** : le notebook la **transcrit**, **coupe les blancs et les hésitations**, ajoute les **sous-titres animés mot par mot** (italique fin + mot clé en gras capitales), les **recadrages**, les **écrans de couleur Locapay**, le **logo**, des **effets sonores légers** et, si vous voulez, une **musique de fond** qui baisse quand on parle. Vous récupérez un MP4 vertical 1080×1920 prêt à publier.

**Mode d'emploi**
1. Vérifiez que le GPU est activé : menu **Exécution › Modifier le type d'exécution › T4 GPU** (gratuit).
2. Menu **Exécution › Tout exécuter**.
3. À l'étape 3, cliquez sur **Choisir des fichiers** et sélectionnez votre vidéo (et votre musique si vous l'avez cochée).
4. Le téléchargement de la vidéo montée démarre tout seul à la fin.

💡 Si un mot est mal transcrit (ex. « Locapé »), corrigez-le à l'étape 5 puis relancez seulement les étapes 5 et 6 (bouton ▶ à gauche de chaque étape).

In [ ]:
#@title 1️⃣ Installation (1 à 2 minutes)
import os, subprocess, base64, gzip, sys
os.makedirs("/content/montage/polices", exist_ok=True)
subprocess.run("pip -q install -U openai-whisper", shell=True, check=True)
P = "/content/montage/polices"
for nom, url in [("serif.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/playfairdisplay/PlayfairDisplay-Italic%5Bwght%5D.ttf"),
                 ("gras.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat%5Bwght%5D.ttf")]:
    subprocess.run(["wget", "-q", "-O", f"{P}/{nom}", url])
open("/content/montage/montage_auto.py", "wb").write(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/7193XIcR5bePZ6ituQJVIuNZgMgIbI5rTUlURS9lMQAOKMNt9q91d3VQBH9p6pqACSMiHkDX6+vJhwO72g2Yr0XDtsb4yvhTfYF/Ar+vnMyq7J+AIIjjRkSursqf06ePHny5PlL3/e3vl4ts/A48sJNtlqEWfzDJvJ++kcvzd7OI+/lahKuw7feT3/a2rnLv63fxtPrH1feONlkkTcLJ5E3CRfXPyah5+186nlZEi7TSRKvs3i19ILvTuJ0HSUtfYmak9VmHXnTKPXG83A5Sbf4PF1t0p0szhI8Dpcxmku9xSrz1mEin0GchXOBexYvvXvybDK//tGLlt5xEqYAYc0iUdqS9pJoEk4TjBnNzbMoWbLBex5qHEeJ9261WngTICVebhQugEWwARaeb+bRJuHXRZigS2lwvjpeadFoNouyFBAvV4RWm0z13WKTCpCoO1stp16wEiwso/k8anvTzeQ0Xh4rhF+/euCdRUkWT8K5t9t91L3YfbzX9dbJ9R8z7/r33noznseANbhXwWgnu8i8KPM6aZK1Ols+5jderFdJ5r0BTG0AnZ20vVXaBhLaXnqyyeI5PjfjdbKaRCmen4dn0dYsAQ6mYRZO5mGaYiCmkfxRG6iO5tMt2/pys1i/9YDq5Vorv3rx0lZ6sQCq2/rxRRKem69fxsS9/QF8b2191/a+antfvjry+jLqtsdht7397tbRIZ49eNTtdre2tj7y+j/jn5dgVuac/q1/mw9oS/56h5G+6W15+Gdme7Tbw3JIAID/0d6zvc8f7vqe8+8j0Gq0AQriLF8vgaWZe6TJVqm1vaK17tPP954+qLTGibcNScX1Ko05uyMuhJEuhJ43m6/CDK10OwcHWrGLXydYxkAbvo3DVGovx6NIYBkZAHpevGTFPXmtFDsyFJuDppTrO2Dlj1Ybzw83k83SlwbOVvPNIhoZ6nbh2n0kBdLNep3EiygZnUQYSMihoKPxajVHsdcJloR0EG1AmiebRRugn3Q6HR17uEnReHjhNvzgoVf+9xG41840ml//vu2Bawnv8KI0I87XYARBqlOANXscjcKzcJmVAO26b9cV9O4+lLfTTRJFozWaLoOz13lwAySbZcFqBJp4mYIVXv8o7ZHPFK3s2m4iMCowBVQpoUjekc+MoiUmMl7W3yqAM74qWtUpmMWTE/CLERvIp9g3xDWPJ9EojZJ41vyKLDSNKu8+8sDtwZBWs3hOjibbhbzJwhgMbZRFF1nkQtKtzNhiM8/iNXoIM8NR56GpLBuAw/V1dlYgU7B8wMNNwlLxgyolLFeLcUL2fCFcGrW8cEYEkMtHyzRajA2keDeazKMaKVZbxDy6W8rzw6dH3udPX714/fTlsyPZhRQkXeUlZGFRLbDx+PVGpZwMFCuWe4VA2vNMDe8+RvBmk06wZFP+wG6kP6QXboOykPKe1qt1Qy/hGuAJ85CuTB8oiya5CW3wKWtZgZdZHLkkaFqfgK9M/QLHYQZim6OyvOAnGDjoJAOK0HKyivkQG3rKkobihQ0pgVZpV8kJc79Bh8ohvOCznWQ1n7cA8ODSn0Zjv+35qO17QRYt1kIhySRqyVOhbxbI3q4jH8TBXeU/nsXTaHU1lC5Kjfe8eZySfGQbC6bRLAQ5jiCwZKvkbZ8vlWNMsYC5IzTxt66DaNBuvr9f/whOAhLERj2JMshBaQT0gBiB7sUKIIcbbGLPQD7ffvPN0+fPuN1dSm++QSwG4Ec/9C1S+1g9+z1BpYAvv/22VtGpqVc4qFR41DsOF4uw3+08ftSerOarZByST0b9JMVDdDBO+zvyJVnwwV5vvJAHe3lX+STX+9t9WO5vd6/o78A2IITRAGulbrcG344FEJ8PCJcLVkFmDU0/cpvudj5hrSugH1PunUQXe8nxODhp6X5/wg20AyKJ10GrM9cv/ke+kkISoZ2ll23W8ygA8wlOBnEvxg6/N8SOe9DCgkq8GGzJCyCz7LW9B63WzxdWSvKdQD0STjiN1psY+3Z0vIiWWRrYL2YostL73kBpn5ChAGGz5bSYfXluXnWOoyzwz1fJNMViGgxbHl4OhkVh4fBoGLgJzgdS0h+2LM5KxeKZl5UrCmAkdv8cMwU5xU99s6DYWJqFSYbW8Dxyn0dL9nFVawodnCvA2IfG4Tiex9lbv+XFEEPBr79ZLaN6/wLDwF/7QwACtr2cBnlHbjMEY79VBx947YCrAqZgUaILvjF0ha8U9EpTF5iJgUi+mF94wdfhxHu6Bil5R+gQFIv+ZqDZKNk511ORlnmxzITRvvocBVboN4yLAp+v5uG41fHNpmwEbv2Yx+OOiveQWTJM8mLLIM3+7qRv0d8iaIFCPf+LMDkncw1xNMlLLEIw1mVkioTJ4uCBlij30QFbno4A0yTg6EYGQL9V4N9gia99C8jNjSgmbmtHS9yhqVvaUHRqG0kYYz8+3ODct4ieJckqCfyn3BnNbFKYKJ+1IM1ho6RQEGBQdlJqswhxuTxtmC/yha+//eIZ5IfR1y//VpYEQBqvyMDY2GS1WGyWoMP7ptLOnMLpztn+jpbj2oGcML+5grzeIbrbVSr2F9E03ixurqvvtTIEf9P1HYBjjat8aF8+PXr97LA8ulvGoV/aJfD0WwWI/Ltl5HZikiiQLR8H3RWE8Kjv216wkxxv8HtGIeEMh6vxZh6ifN/3i4X5ujS/Rjnh4UA88U5WyYqHxeOorHowxDE5WcVpDA4KwUlkAV03xdI05fo38AZLxrZYXxdKQbJmaTuLK3+FwxU2o1kFetJk27tUPFy1/vV3/w0Dn0EEOulT6io4mwBVNNuxkI1zXK6hNBitcHybjZJoveo7lCvsV/uwOG8ZZANVfcV6u5EJV/5xJxlx7WFRQbgTINtYYxBew/kInHmxzvrOxHFjIoe/U+MUv/QIjf9wvjuL5SSNI0r/y3CeuthQ1tC8xybYo8x37A8Nc2aYkrO3UhdSZmZ2Kg15fU2kvW8uyyzlTtPKfdbtIigvyuaJmwIzELh86gLRLhb6Gnq8EWXqvg/IHvlFB8AE1EAjUk8DzVRJ4C83v7/Q3Ja6wXlDZaCe/Ypv5x1+IcMSOYUP5BvFlaUWwCd+uWIEnzq/r24cTS6FBan0k6rk1RpeqfRmBLTU0p0SEU4rkxP3gcscMJtydttMQ1+kMRbu8GcHI4cWJIb0MOfmHgFPKLje5Pup1O3rM9nwlUYEwJypKiMuceeCezv7rVK1/6//+ff/93//J897FYou9fmr3+Boh3av/4CGt4X7b3vcvuOUm+oaVA1tzlIUmguoPFIqQH+cxGmr4/kVRPpPJ1l8Fr3jYV0bfnaBshtZP596WALxjBpT9ERq9qbbkfv+9QPW6jjkbcbbt9vSlrs8Pz/heEk4eqbTIdgNA9pzu0ChOIf+K/Eugc8rLlajh0K7lt9C+JyOpHi+Hs0yxIeRMH/WImsmuTusPG+23j3oBzkpkI5aNzT3vlXYplpzvEojd1F+CLPd2vrq2RFULq9ffPuNHJh96AtJbifRhh/4pT9VXDhZyMdiIQ/1L9SK8mEqRieQH45ef/uKxxmeInx/LkooalzmJCSoRURtokf3KekQ0twCkqI3XUHDCEQtY2hyEu8NyGoDFYNHZb4Hklpi+JAz8Ceep/I03VqgEJUBJGYsxtibb/A/9w7+Sb231C7RYoJ+VHEAlbx3/Q8huvAm3g8oTf3CDxv8h9oAAfBc/wthCqlsCPG5udiipjHluxT49ESTtliwoT9m1DbFVNOHZ9SwhFwwLIlxrvFUVlyWXP+BfyP+gaJoDS2LyDsiB6VUapOiqZ7bmvLX5AStnNHMscDAM/yP3oElKPNQ3GPX7JhmBBzLQF9n8ne5IoLwPOY+A9xA6ZRQG5biyTiO2NQm28qIQ36L9G+KUzuY0ArIWKKbs1UMXat8YOLwDR/xJOZg0QeGGSfYf6GZnwlpT7YFO9uYqR825i8nZ+vNNhCzlLfTbUw9/4AYttEFxr6IMTXaIFTH0KoJjkRT7CndejwR8l20nECRTiXQG0Azjd6E3hnOFqTjLejzMIzF9R8x2gXYB3ona5XhoyoxLTO7jjYZ/1wArfoHUMX8P4lApZ0UatMsgHbhy6cvPnsp+qMPJmDSi6EYkt3UzmvDdBeEt4X2liT2N+8n9zolTz6IQLYsgbzlkgFxAWdcP5lLoUKxOQW5M5VTZQlhelygjjXIzAZlmFASdWAGCxJ/8B++P7/+/fV/uf6v1z9e/+H6j9f/eP1P1//9+n9c/8/rf7n+0/X/uf6H7X/93d/vDMlC8H8GBn4eJWi8A+F4DosnNrrf/T3fbvu2QygMxECZQHUPzeBEdJA8O6QUsvInxUnEeYitbHtOa9D1j31jFXpidIwY4ybpO9/TbXveyLizk01e5QqgtayAWIcqCMFgn3y/HPolIMjVfHf3hjTg931WlBbKKpUQ1hp0I29MqygMfVhNFxRaJVFdJyPADmRW8lIt6mfGJcVSPBO9jpSuHeZF/cLfYBJlxdeCoPN1UecUJaQ7aILOrShvujhl8Uofag1ZTtgyxkntAmQC4q/d+au/ftLDzj68928w7Gp7drOfxhOI29ig+zrW0yH0hoE0OegORSkjzasg5reKFjCcZn0TmaNSF09lkEPjJBCjC+WHMQ20sPdKq1S5h/OGoy5VPBH5hNQjGxC1oTnr0l4CeUuaVt4wsA0PIaXqUVd2rxWlj01SnHbBDs4icpa+Nm0XoDuNtkxtJs3UWZwY6EUIZT3SWUR6L6Of2LdNDnZ2hw6ZuY+9e/1Ss8bQFU9FWwppMzhvUaVbyOO2tkKVraDzaVN3qXSWbhaBVG8ZtDuEB0v3Wpp4BxK2zbS1Nwe8KZoJ6LmwwxZa3sesdV97aiID1Z6eG+2pqjAzKiuN3tQ8oSNEeBFAUf4QoKFVKOL3WqbcGuWgHL8q6CwjZqY3EJhgDJhRoNfvWwXnJUJdWwq31G1aJc+JYVG+jQ8+NYxTdjJaZSL1wnC4oxSFKRKnn+sfxXVlsZLtFKI3kAFLEZmy7IlC3cuf/hlaemx/b7yAra4mkw2ag6HBHi+wglSfE+m2yFLrExgUopb3pN6dCOPsDVsrfkJKebOiOFfpHwcOygspDxx/J3LE3xEA1uIuFosISAhb3Mco66gF/DfLvBEZFrvLberFsAJuvLDiUpxTqzdlKe64VFqVFFFC6sqRWmUOqbQ7icGiIik1kfcTvq9sEQOZ04nRvwMbUeSrlljR7xfrQKqnLDINTNtt7zSCmS1cjKcQbXveDs9Vk4Efg3m6W08bs9T3nJdt++NNhW2TNQRd79cozT9vvF/z7Ci01apsOca5JypUGetwqaq5FCadN1g5u8MtV+OfP0WpgtuawYsYKNIAoGNToHecXfyh+QWuAxsGf5p1VDzlXtEq46mG5tJGfAu+3zNGQTynNJO+MjnJyx54oURwIbOUFcMiieqwWrkAJWqEAjfEGrEsjdemY/kekFZ0VTmVzbrtdfPHOBgD0lN3AntVYeIUs7HkPOdFZH9QGUJmjCUuDIqL8WHXO46CJYWLvqKkLo241HDak56GtULk/vf+UsTg/jtlN8vSY7LRXhNEuaAgoA9bT7T2bomWMTRy9mbLlezWFeW0MwPWcOL5nTdQyBjxSZSgOvyUFCOKSMPsB9wU/DKX+QWsomdw+JuGuV8DDq+pOHQdPfvNi5ejz7/95ssXT7/5/Jm6R9ld7E2UUeyWMZYHdPuiKIkwugSE2jKhchRwNBNDK/FDuHib5ptbGm3ieb8CXbHNfWE9H+kCU1L9QibTrSfl4S2Nj0UWmugJiwdfOBSpZivfCBUXgIvq2wSiW4CVN4tpSs9VVDgNkx306KgZYxWKD4oxkpjD1FzcFDF9zqaX0EZGmfD6R9n9Apo65DirQIqXUwL9H/ZpPeyao3KPh0SomuDKJjv4cZhMqY0L8SNZxtd/SJy9dUbJcaqSpxfgy5jn37yPcAzkwz8FOkI5eWMJJtQJ2h3O4AubuBhcyFwcoQxbihBiBJfFCG4BUZXDkMvCSa4s+yeiSpV1hZdcr8MqS+JiwDoXNEVnssihWAT5kY2XhFYIS1Zi1aK1I0O+phOeKCJpNe+8O1SOgtZ39xpOU2b8lhmgChiBxUWxH2wSWyLO5fLSwBva2Wo+YZ1iV55elLFqajvwnYtEodvpsHCWQM2hoylejaXYuZrocykclR1zf9nI7wwKhG6FcrbUghzNHcL8oBAqHQhvYtXCUCMLIjWVd/XALGdeqSDi668xm/sPb22z2MWwTJMsSstIJ4oJogO8cBJZoWhf2ARn3YLzaV/7rBy6tW07M76z1GU9i0bQXdF/3fJbzUeJJQ4C3OJ2sWXRxQoHi7TYssTdSh4V+5aPCaOD1fSiWRXsC7ukmdbuEue6SzhzySMIh4xS/MAvMyQ8MN9uaFwZh98T/NmiNMmUH10ZUv1IeVbqMC1607kchsyJ6AvpAQAuJXov4n4/57xQpFIYMr536hwdz2bwNrBml6MIhxQw2K/DDIqYxF0ZYXldAPvObEI72M83pnBgcFeWbElsb8IWyGPvPQIVexx7oh3NRADaFfHlwbAmP40HMtvkK+iVs0x2svewzk3gSxmelp6+GTswjxtgFiEqptqjgpZAzQpvoDR6A7cVccqqOA0Bqk02UpZuhEsZu/k6VjnvzXjQM69EknsT1rgxAZDVcyA1bEN4tC9OTW5H9VEDKZYih3axzHwtHcmxjvrYfHP2LseD7eX28Mq3sq/b/F0s1Mo+Zn5p7+uZva2po1YTzGZ18JgiZpZbZrN0yP/8W7oiwjj826cvX3whkgwZl//NT/9MWfYL2YL57UtQCj5ei76I9irIHlRotfj0cytm8Mf17wrZhL+fK2j4dlQ49vpWXJpDsInSUSHWVfYPK6cN1gOwLHAhVXKsDRlT3+g8E3rWZ+ucPisMxZepwmvhQ0OyZGHHZh4utbHdbpf6mME2S23DCc37VdWdJrdr+lhqvvfT/7Jsb+3QkIqN7hOrDBEQ7bzJD/Cy4dDZgBQRFlMjWhMgnpbwwmPbRavwXFSNtBVP6U6Fzoj7Lv/AbqIfRoXNB1GDrtqZDZVic8lKp6tBTSN6f+hOxPd6vq3GXHXw7hWBAG0r/6k1qaAHatWMMlK1Q8ZzOIguOhSMESQjUvEMMjLsYlyLGRXjkZotNstc+sWKmTHgxSowI1FgUt6kn6iYQ0zRQnCsyTXrEzNUq8SrIMCY6Av2oYWhzqOqKpBf1CYOREoBIT1qDXqPCnHAqGsVGTgkWMdKqTjYHxpVfNtMuj4+GFYefFJXhGiL79ktTNm04qRZ0zQKiXBnXp/ky00UjfxtlppuwMJzrlrv7bZ8EiMDt0/YpN1P6mDxBcBS1ZXKkY7XLetSNhm+B4BbldO3Kqj/f6DnA3TRrj66oorOYSl7+d5NIZ0rpfMR8FBjW7xBQf1nK6mtapovK8hxdNIf5cKYja/hEhZjCrScQUpfDgQ6XP/IcAGSv0h6mZ6TwZnmNnJC17XDWAXR5fXsyEsfTHalffVnajkYbKRBLKLENVpzuJ1CyE80FslwZon9aXuHvTyCreIH3qjuFcWNPanmx9fdaGc/N8EcdprDt6oqpcGi3rSoZOuKITXkFMY3s235OzzVlZQpraFrMCqb8NRf97fhfFPy1hUj8GouBvgMelTh9dxlwPrPJDbVOvncYBo0+OiL8WRhzjwGN/cY9PEoP1owZOXdaqn2ddNtsFp6RpWTB7ECuXkYW8vEfi2jinv+XfQSWk28sexkySM9nWGl0pMGjyEtH3aKyLnSmmqs0DfDfuK+Zv+F9BsXS7Ouf5RaxdIHn5CGewZOPUgWj/QcKe3DwS4eXhUz8k7YE5srengHsSmfEsxAm09sH0TCYccJ7Gu59cZaD9KYWSHv3GHfK2pSbWug2FRmpgkkTAbLcSoMdMA5nijmxpZWKge2UgEdjvuobSCGuiZ/rJNAPvjO/rhlHlDNzsK7MqXhTTEg0Nq7Mq1JsVY5JkS4W95ruSPDBgc+K8pg3sXusiBrgjaiKdK6bdYH+Qqc52m7EMcEeHEIvxOnI5qUTdTs24naLRgWhPli2GSbtssH9pei2rAs7fi988djt07cE1FdNaFXzu+F0WC3LYdIlK7akxbi8NYWQ78iZnqB4zc46ZC8I39S1xnmyzgEjzjslENMZV/85KElMj2tL6zqD6/qx9aJSLCBAqSM7J6pQrXVXl1XSHRZkhmE1JlmpD/iZ2KsBI3lFCCz5+VIvwRGe/QJ8alJgjXc5wRRy8NZHMTer9RcxF/WsxXEGNCLpFXRH7JNswGYcLti/5Z9lwUcbeXAXwkhZk9UblhblK1lnk0IdrImoJlLqxq8KUy8GnNqXv0MVbJsXNUtsGejBu1pRQWW2nGoSi41vbQD43vUvTeJ4h+u6l4i4DAx4qa1vg3euJa2N2LHBBxMqhAtgyanmFsU5vvdmsa87ObxfqW5EBkB+FS57C55RSWAWLRAOphPvQMsthsL3qKaN+zvr/oGKP395+nyO49af+7svl+Z39COWQLG9xO7YiwnaC0px2WepyOQooa6wLeBMs1Pf2rTzSHK7Dc08NOfxE7Uo0vgWgKbIZTnDYnyNF81x3HBUzlPBrIW+aWDgGO485Avm9eD4xiLr/hRttGryVibaxGXuc5veUHFaW1O6X21q/Jt9Q01hEaZX5v5QnTlBB7L4jCEjzEZb8zmSrc6LtWaa2qjVMjuu2alLi+UlJrpL69ti6UFFdoG2ZoYOLoVKlxedOB6D60Vo1qPO4gdF0dKvnpe3miP4wY7kJ3eD7MDveOOLKKoMUfowIoCEP/oQ5/bKorT/UVm2VeFWoqhY971mfExyElQpn03elzIAPHSyI9ot+0aQkTqOMhFtmIpAtkyQKFGIZZzY9uHs0PLWgXOy1g6Txu4TGw8VU19q84Tl/Wcvt0OSLsPHK4gghPZmgDVGuzK0PWXtCDUzxwIrQaT2PNCoDei+nmaW4nE182Yh8TT0Ecr3OgZv+BLpD9+sb2r0nmpIZdOsICSEAlUNAOEHtLypc5SSWaws7RnAPUCOuzUcpu0ChfDHEXPBS8HBQFmqJ8xRcpzoR6js3meH4ocaiKqOJlxQafKufbatu0db0+Uu5KW5NdsNB4WahP9pbaWXzNZSHdYdskhrSzXnTnW2JrOw1yBbZIWeli2dCRkaDI/A3biWkvjioGxcCOStitx2xOKPl1Q7gTbXQAU7OBnmfBg00r0aOoMWbBAF154+4XjNIi5ZyufdeyksRXXKuKPNnkXy9I4EhUol5tWKrmJxZAs0XngovSei24RcTmmScWPhrBZYmYfzmpVspEB22wAh5087w+WbP3pXqtyRmgrlorVbHzdBCUVbqrg6vqgtGoBwHnhV0gksOXqkPJ1pzoe4YH8KZ904qQETEUjP/HbLBk8eS72Vci6YmDFZ9lm4Yvqzr7jXmjPHuR3sivmuTrMVtjptmywbZpM9qDeEgkdC8kMsCSXDwxsZec1FcRVw3TA1ZCB/6qYXkZS7jgqIj1yXJWlePNamNXPz69A7Z71cmW+mwCWygU5muaiQeASMqiI1otJs6AqNgPOkrfOeRlTmSeyQqTWJmKUWaUpx9nWrSv1IRpko7yn0fjtaBkuEOhlnzjal4tJtM4Q3sYPZoApkzpzWVXwONu6vZ4p1gC/jLcAX+dfUqtQg2LdzWcGI+oXnWhEwIyHnvF4dWG0/CabE7npvp7koBxmwo0EkzsXNffHfM/aOzzB2QdqmpGcPgbHnWV0HvgvITcFbAOL1DCxPMVYh38CqdTqsP8gYNc7BFG/sIMHqGmHgGH3Z/Rtn8/7uw9Ne7ZX+YS/M1UYgZO8rPNcopXC5WfzTRI8tpvPogzo4fPPnjqwcvcy/7Ugy8eLDmJsw/n6JDQAK3WJGHPHVpoGjwZuGvoNww72HsJTvPQnH1FHABzRT4LJyCJp3l2M8aKts5doVzKJIBhNq/ZKVpbZA2StjRh/OBoxjAD9H7ru8BE9arBugOuPJyetRtawlIG53skn5DCrtMNgdWjsYX9LgwllrskJdnk3CBgkPSLDt534948kD8b9l9hzwuTtfa6B9P4R9VPich3O7z+PVslxHHovmGlw0smymX9bYLJ/f5Mm91OcLaP7M2nNLqn7jAg729z/Ah+/3Rwx69aO02gVzvEHwvkU7GLufbaaT385GMHId/IW3fjv+ayzFt9GfhmbL4l9khB2qKGd/GLt4rcmFWvb6bBfxuXmJ0ijYKKYctKhLkWpRrJV5dypHN0TlF+WA3yMIF10UfOP0eyX9K4e+w0ewnlFBvL0850jx4TdOvxnF7BaE3l+jpVWxZe31Nvkg3tLnd6+lshrQ6R5j2nrDt7Dd+/k5tatM4LblPVakHRntAAelj0Xsg7IV5wVROzQrGji1Z5nP/NVBMkD7Wol89RotqRNwwMLEdPlrcEYA4hEh22T0s7xXHgFb4AdpPhifR4/bK5Fz0Qd8ZxyEtK1QXQgkisNejroWTV20+SGkzxr4fU/ZYU3wekoy00WD9omm9Fhx02PR50QNNlOCICkaHruSE1Y73UfwuNCgjRuK+UJ5TmNi+DY6rTb/MoD2rDuSN/YQu7MIJ0PAjN/etDkJIphF8dYo1t2fPsq6sJmctM4Wp6SP6DlQa+qQi9OuYOgCOBsd570NBCJtvlzusIbS6OhNdFP52/YTb1ZDe79UAjV6a4BygKZc6kwF8ccQUNbD81qghKHv3J1HjjTujZXWrHOKdp8Hc/i1a4Ul6/hkbgIBgHzqz6iDBQcPGp7nzA2LHiIbwfy7QG+PdxrtZozaDW0qWE69gVkNywAE66Tl9GHjS3GSueBI13my1X052nmDmAm3jLCn82iNx1zy8/S1m0+d4IUbh8smWOvWWsbpWszsm5n7xEGkPfTWHx+bjTjMVULe656a5FSH872PlZNpTzZqUbKuo6f5zwkfSd2oEcHvRuHVD9I52RjT73sDBqL87YwJAFuvwJcm7BVDs4w3yWS4/RBWZFwYkaZ2CEmVjOig5RqZpjmaW2gbw1ev0JBigX1HLeshK7uVw1XArYdz/xEAM/7r6OJWuTgO4qlcKe532QFY4NBLsGeU8Q+aVniQE/NqBfe7CC4jZ52tIm3PFPPT/ATDdmHDERrbOiC5qrzc6WOWom3fIu2DFpvsf5a77NBzqtqLCrfEeq8xKReJWt4DL3mI+ifPgGHOOg2BmxLWXfJ37S26+valQXKS9ynWGEkNl3Gdq3euk5La/QgX6N1WH6Z1XnHlVlflfMTIxA0rr9S0Yu2rI8S2SI5tnzy0Q2r4sOI+Bch4GbiBZ1J66Ljkm9WEIzgc5W7r14YbR+RctEWLZPY9ktHS6opA/6BKefjj739PHwYc5yMmAk3okr7LARHkEEAfGChXSTJ7Ut2XNOnhnLAoFxVgNi3eTXNeSheRY5Yrl3VDsTsWbKCXphN722Lx3unaa4XHf4aL9KJGLUf7YkhYZdbS1TvXbL6qlWi01WzPPElvhAk1d3OQa7xRos04Dx+/NgJfQcZLC3Z7SpgOM2fx1Mw1o9Rheus/A7pVY5PMvPSyT+3x5lcIMlFGr+LgkBbbpn87Z3PXrx88c2zp4dFDSGMorMdwKJ0/ERZmtPXDltS2r4nYOw+4DLknEcGhjK3s9DkU4aR79Zf70Fnu34biJZlr1Cz8AfYDeR6Jt+H45jfwv7Dfo2m+UySSwdnACJsGQBunPa9hnn/+WrJykUCQfp2mSGDdcbUWOAVo2R5rIaDhMFwi47NYIznwSdWUzeajque4rtdrqHggqrybl5ObLYjxtVFgYkInIHCZqCKH5BE10aZ2XBfKUIvpYnCcBytFmq9sLWWeKvu1Hj9DlHpaWBUmDjESXJ6sat0SdRTkx78vvfDVsn3yAbrltWse2Kqh5InFcaBL+sYj2YTWhPve0cOCZ6AutgZweXbHel9R1xOCUFeUMABTc7wgpWeSEHzgC9lNGyiX1SspUlIZ8yiuFqlJ/Q2Q7JhKNHW8QRfKGDPuv29h4Ke/kNcYwAcxfS/7e/s7jvoJQWhMrrFQJ54WdU61PYEu7omlmf6+hyxNdhQqU2PJyDHDHjAN2GXex0SqKyljAYS+caX8jbnH6mc+R0yICUxvx1yPCCnmGYlDyRJrYyBXsDovxSEwNYBDY00CMckV+GeCmX/Y5YmMeqQW4IWPMJAHdTRspzj5EEFJ2CRuxWkhJY+ZNa3cvo4oIXrsQRuoBSyLgQ7RAhobfdRMVi8IgHtaSkhIXxONgvKBzNts1VvgvF/hjVV3nR3rbrUwUcVHRU00KmpQADEO8gtRapgO3ZM0vuHbkMXG8ZFIs6qYwnEXmmGwksOKMCBdfGVnGJbNekChq5Z2dA12N3ffUQy3338yUN+7h3sfwJH7scPd20sO06Cfa8JXft3RNbO7uMSpo6jJag9GQFjQXSGH5Iirm094dXkUsFgIC9pU2hZTL4s86Yn3mHSwK3QWaoJq2Vdw2RWLPK2bs89S7y0uZkZNA/tzyvXZQwm+wXNZmVMFiNx2B3CvRERMyNVExBkN1gMnTiFNB8d0ItiZmyupiilSNCrROZx58DfwU7ag/sjWynCViPr7oDzsXHfQlFHENDkCIHLlHaE08mfCJCkZoUwKzONmiq/7OxqDBTGIJKNQai+zNGu9fPOXg7SXiTer4S3J2+HSsjpD/RI6eQkDLhaMot3rXJPq5h7D3TuJ3MGRqzJ9yanweAljiiJBJINenLSESoSJOPpzm6e5uo8hoTDm3M6TPWcm/388zF82uFcde54v9DStzSyRxrsAWZ5lCJho0hKxSNEORnqcGcVaUgR6BDJS1jDU4pq+3ufHHzS6iBLLK12GACgRUb6TrYav0W0WfBLSCRIkTPdqLhgdCB6rUZk0lEaurVpkeyNQp1kswxgoJ8xSlpi0HbOJPsinfblJ/TFcNQfIQogCheSO+Os19U3J6vzEdO5xcypcZstQyv3BYFtlSp7+myUArgRUzf3k5VGL9g3iKlI9WF0e+s7qxnh4RVKzB7N0WL+kYqNlzWMMGKIliahrWS8FCdPENHUJtqgiMPakuwzlRjggW8HTMck1yA6xeGd4+ABHt91NNb0vcry5SppYadkI1PjYGpHOmL4WXP+fiZzs3iQnG7ptGJgkx4kzalorFOAkFewTF0L8S95GkUWhYAYRb+XVy39bZELIyVLdvOzCvk8XrbAH3YfdckQHjtsyiACHPbc3SP42GwC6q0gM1GiPmvxVtA/hAhLpAbPJgo8WX9qbo7w75Bm2pLJJD3rr/vdD6aUPPOdDcQcTTe8l+kXX15h44jNEkJSr+jiTsvh547TnTDeByPL4hcImoJFxwRMaQ5hzaylfmXYW6x9WYsE5kIWKA3otRPTJEmNyLxPo4gT4ZrqxUFerUWGmjLuCLkbqMaXO9pMPOo8D91yb9frbJnILO3Y6xWJzfKdQ8PH5Zo1qERbJoRpG+l9JT3mZr7xnug7gZPsZZMil6kmQMULzR32uTVdSVcmOwtDMvQmo/oVc7xrzYvv04bLNJ2ZXFe1wsVr3ulX7xhUP5dEaSEynGUmNleSqS63cfub+BGLFz3PCZKbRdOTmYFKUloY5TDbG7nGbpwnOctvsLChXlrlDuFeOguSztRMUWIjvKYrOGdE4qsuU8v47I5t2oT8ppA1T2E2TdLAFIcQQ++B0erUyaiuMcT1kC13wmSyWtWAAO3NEepUiOiXWJgRWiverXm4k7IzPhQm2qq4p2qLDEmipg/6W02zgMhhla4oz2kTBNBIyiUxMc8+KBqkStRSDfUz/2tDUJfiRXvlBZfW8wK3zkXiuWSGdEWvcCVJzAdcyU1iXk0LpHF6IHU38/ZabQ62RYmlz+cGkflxr7s/veos1g+cSpOF+GmCES7W0bEwOFx2NodYPS+zw7fK/Mjy0BaG3Ovsz67kYabPcmQUb2LfHiwK7TMXSL9g0tUptNHVVQctAnqPkO7MNHH62Sy+Q+8zP1xu5nP43vWT/uXR4VVvMu/zUoBo5bqr5o2fzaRSOmHiz8vvrnqXX131Zpw23OQQH9O4PQp5QQpURtzhwPcnCZWT7Qnir2yN9gxJxC9xIeNVW0TUpL/bNjvj283Zg73uumGn8HdC7RyeJZRqCQAAboemppF9R/PwLdZ/akfR1NAiXEvWgp6VCPMHIR7IAWdlPBT9XXnW1MqkJ1siXEov9nB/DduhrTWSjBbggW95vYI8niQyKbsHN7QjO2cYTqTwWH+CbZ7K75AExvD9IzERc4Nlqge2TJJ2KKe8e2PShLomp5UrHAzTsRp6/izRl25U1cXKZ9DmyxGeh3mN0TFsSJ4aRZxsOjetM19e8zpR3zniyOlG3vBwg4Ex2/UU15b2/U0223mkp51ZOY5QvUJ1LFWvRznKID8Fwze2mUBjXWRR3pYcyt9/v408ymTd298vDSwN4s+dVr2KLUi7F+p0p+GsyIohy0zH1px6aGei1dfa2GJ1NpurvOvfIwXptRBWjhjW51SvW+0kC3DAqEA1jFrQ8Y4E2NQp3jDDZnarFhLt0SpzcfXrSC7Iy1NcL6ymwEbdQ/EHFSTPy5Cx5e00PkM2l4BW1f2DLv/h5aL+8sC8SuuvpMmSFO5fnvS6e1OwkoX5TOWzfQlzFJm4TQUbyd09hNx4FZe0OAXpFQfrO9AePWB2S5R4XPgnW1/pYWM667KL87HNfNCuv7B+/gyar6e8zrPKaGIm9wcsUGFDAGc1RM9dJsDn6dX3y0tnhkNkpNnBjcHus3GLhba9bZt9a/t8e+iagLdpAmYym++XXFJ5lkjcSfi3r0e/ffHFs2+p4vFli22LzHSmnw/08zwaL/TBqT4Iz2LJH6Peni+cmxZzAVrMREzvYgS26bZ7IWNx2aO9q5OSNhP9a8CISepOCQWBZ9wjy3dAitfVTd6ltD+6WjX6p4m8Jt9UhlL5iCWLyXafaYBBuQ1z66SxcA7yWyiH5WJy34iWMadkXlEpB+LAF/lPdzKnSevqxgtlUpJ/kM+MTawp9vSaOya4ohlWvDi236YXbTto5FqMxKVZRUT9S0WWOtNbBErjuZela4BJwolrg8xFVOqhdnLUtuT6UlgV4ejvoNkp0CqrKQs89e3Q6j6ZZlg3e6w5ntfCLlyUtmALXPLAI67UDemzUiNxfwfYrdVSTO5ebqhsicV1t9vsPUnQSibSqrG1pXn2anZW3ZWtJfUlUoP++2+PygC+MzYyzUBBnT4novHfR2o79P4GB7zPwIaZ38692jzMozFzpnNOhuupQwPHhzG/49CLX2XnBMz326442Zpx5zbeybkE4bSdV7mNd3JSj0F3XUZZ2KKON4E7tuXPf4P/wZpXF/1Ae4eZXwKZzgWUe2y8PMNP/TqFcXk0U0/xuqRceSVkVJvuOwgcsd8urWgREJeqn2n0y0bqt0LW/plSu5XVm/oxUhBCBJT1iDQcX4xmC5GJEOq0pyKypCwWhU3fxciLV89apS1WLBCV1f8xL3FvVcJzLTPCGfO0jP3xRrJ22jmweiKMbBqQJkmJ9d1VLNGbGeN780J3iTLL4SilRnZ2hpxJovUaDeWvG+nIDaUpx20Yii7iNorwilpFZphE3zM6/ZNZFdVLIOg0crK0EI3D9UWQM3U0B62ELMxyjoLS2mi+ObWYm1Ns2/SocFdMKWzU4YUmQkuvRF0yeMbeKCjik5OtqNC+mVuU+7o1oYNjkzTfqBPr2jnjQYb+TJhJYKwlJf/vithX8gZXBw9GhDbFALr3PtgEEtqahtEN2+anRtyV85CUouG2ilvcNa6+rz+OE3WHyPEoHu7upe0SzFqJbimXcLPQHZe3wUrBKoU80Qrz4w6ZSDA/ziO3HPosA10w6v1ut+RAtC9293nO9u/zu+wOrdbNW5w45n2y15VlbXeT+04zTI9woJIPtA5lqCz2HKiso6sBCs1/GEzS7jSyIoFMn4YHh3a6nUw+O95+pzBtSG5gSV5CqHtlWE2bDHA1P0uthXLTme7iDZHSvBPLkXqd+3yXKllqMHTlCmRXNMB6QqY9tD1lrjRIBg29wAKApIl6nVURS6hNiysnD0lYCuqcaX4UJysDg/EwLfttOY0cFxlou5LM5zjPlleEN7RuPLbVwyDK2aDLV8mjQ/dQApneJHDT/JiHnXJxTQN3w715kI4zxbaV91tNqxMeR8WBoDW0s0pbzEZOMtSK6jy27W1hyCpLZofDT1uYF7Xa821M0Yncs4VcG5PVGS8Y/JMnkwjuI3nQlRJSEH1KMe4G++xWVcplBcq36Ym+O8Imc/SV2dLFs2oSxfMg5eqhf4Y4T+wZNzfn/Un5vbR1xh29QaY5oohyRKmGupP+OJ5s8L9fcL1IByU2E8lQIMFRI+cxZuzZ66dMWPv0+bOjgrjQJXZ0v001klNgUG/CZBWyTsxHFB+ORH5QmoXRUpw+NNM1j0c4kxybfAZ6wefEZjco9lfkU4QWY8QZxjIKzH2GzBdVwIhjanAYtfTKrsRc1lanCJ14OQd5RTvMp5WtlA+gtmTN54laGSCDkoroI4NPDmWg8A5v3t3dUmaD/zDFeik6yQH406rzSaGgLjTween7lBpV7e0kOXNH0CCcS4tsMLaGSCtjgyDaHyTu3ilPs7qL3C4WK8ySFJIqsmKEO9SyqBm3RCvdVuXsPVI6CMrUQ94j8rPbAdmYNO0+/JSoBEE/qCYvLXXL1itCutvIr9lwJcRAa2JU7rw0CPOmYEmUb3LLL8vxWuIuQrwLZ02OzwnfTEG/PhCnDazsYS4wNYjeyhbB9vC8UeSuCvFOuzqt2WI9OiubIYUaRcG2ZTZGZEkSTVSNwj9Mvf0eSr+LG4NZmd9dXeAwKe1YywZPdfZ0iyXTvt24coe+Gq0vj6qw57YmxaQeTONl8wJcjsvKbuOzRdAh6r4puc8sHTfjsbPQxL1lqQzJXX0r+tiKmF8JuJS188ZJKWQkQDJmle/0/ijNaAHd7yozrrF1snxTJ+i1OUkw0Vr5XEZATV4NRGysTKaMVS3dmtFNuawlR5G4kAl+qtFfiTj7ucH2jrhnhDl1v2fiagx93FJVYqX3+TxvZlzcUlAWungrQUdM0LatjpqrpT1Z0wJPKVF76eRZugKh1HivKYhnbDdEgic7aMZEQdLtDSE9HXOGrvGwHLT8Wg+JUbtFV6nxBM05HqyuQFtlioNd6gglQUOrIX6cA7hxcy91hpIdo9zN7hIWbvS9qt9ROivrditZFutC8zslUMkDORTPS6vG3Hto9JiqVu3u18EuFJNGI1lVRao60okiOvrOJAGanKsikhFFR9+pavKJqi3zsl9JANdD0U06pb9SZeUtWPyLaSrvbqkqQsGdY1R3N1899jjVTMkaQmljpTRGrIjZujnU9D1BV0onmdy7A59ySdGaBzTVN/5Cp1C9EOewIy+hvJnAhuWszvwUzUdlhiBJqnQl3LAKmoAvgNB4R07+3n5u0bD96Yjg/X9L/OUdlvRk94Z1bNGhsvzti/nGYUjltokazB8YVYeJIQy+Kr+0lg15e/uOrduLbH+KDhcbRoDpyM5sDJYmZqpseHGdkt3kUIWmTzMmwjU0UCH2oVwR0K3mKRUfVd/TXc27XF7dv1yOrwDmpd7aseSY+ORXvJ1khu3gpOJgkVvYa/TH3hESUe/WWOApGyxNKo/K0Cdz3HFGDWn+/Dzk9QJPbjxmvW/PcradIpNe6R/cC6fxSqE5yxw3NT1SSyzZyMaS/ZWNanRMbONVoukJTOqyy8zZ7UfObl9ckyc5e52E/fkbbaty8KgZ1Y38U1frr26nfo4vj1m1UQSIknCoaaJZ9QbH79MeSUaM2tzXs2sMq03by0znlRR1x9AWSEJJG2Y9sL3Y+x0HPaOXlBS1c2Es95GJajjoHQzrzgqT+fsx2dhZ2pBw94Mwy9CWMlqFsnh4d9CuzABcpQh9aedZ5bADmMQaCIYpH3/wAMvizLdBNk5EjxPLgweG5G0ac51HTYQ3zD0zyf/CMRhyPUfz6XtTnU81+bfuoe+KLHN548X9TwP4nQ1DeC8v+pfvBtvh9lC0FT3+GJsfbVjjsjXc2l69Pto5ev308DW+tP1mzgofuVk4jfoZDHm9aZ9CUds+wQQjUKF/aRJeTnWHR6ygdCmF9wchPEKGjvygaHAgdgvwXO4/sbcTcWjiV/VELi4yj7UBW55hujPjMdVf9i9JtIq/q95ZH9iAP+AN8CoREOa9hw7Mew9LuAACEFreBz1BQcegyUuZesXkfLWZMvCw/4IBgb3Xr/o7e72Xh0/7u+i17FtYafNWT0PMEC986e8NzoaDs9Phk3LlwS4n+fYWzlZzkBMjPB8OUotecVFRzlPRRokxeHaRs2Rj43L04qaqUY1peMp8JUvQ39nNLcqmopMGd1H3Ez7mPoPD+qEka4XaVRIvaE1fg+zFj7hT8iQ2c/1ksFfQ+HRh6HvKJPvOzDTS+Ae5e9ZWxMw3SL087Og3C/NVu7JIDm6iOT3rQ8nWKUgO3weLbnmSzUTjMeefYTMA0lyiCBGkn53gAySpK7KnpvaDHrCKsDCkB+wliKkADvr7D7uDRWPbIK10iHfhIr4A3GuOfL+nYbQ4N2Dt2NCS/gzO51kbjxeQmZK+fKLjR48HIcaXU1dJ3FRV9xOCr37B8fL0iXbq9rj35/WYK63CMtcWKQNeYKH/M9wwPzbELhorSaMo4fRzRp14M8fJ1wBzkzem8ZcuqLLRCfdWn10ZYpOj5p/pXxpbDZX7I2zXXEcLz+AbxvbhvqWFXGYAMF1zC+3dnHcURiuGwpwheryWYfTbI3H3vyG/aJPrKVYN163cS8Q/pufjfOPOBbCtck6smf+FXstmsq71vEupsC2zy511d3YFG1NwaXVrg23Rh8mVfDb/bW0VXhZ3uUpW8mMy34vBtoh32yXJD83UTaEt5xJ7baB+75Gm2nGz2J3y2HBhpUtRUzk3oNZzf19Y+c3JQoaYIW7WcDUy/o8GoqGvQ2gyfNobBcu30TnCgOddYvBQFwChB5292VWKJ7mfqNxdcXWJnu3lkva2bziJGuFAr/3DbPMSN8lQOxqJy94IF5YgGmlkjhTmVlRcrQMxw1xDGVKJaZ90nibHG4Y7v+Ivq1ALEQA+nY5C8856R/I+kuobpTv/hno7O4JRZm+er/v+vzvCfZZyjxbmDGtvhSBJpNFgPBOuPrSXYrduac3snk2Q7JAlUN1uUnD0/ZvbkSyY5aKNLTI7ZlOLxCCC+okzlk8N3g6ZadN40gSur0c/FD2K9XTRLJx4aLJxlnJx4jG/tNxrw/KY1UAsEaFclqBx3Prd5F4zV5YjCUXYcYzNcp9P/7Yry0z8k63FlFBbxem+xk8scOJLZCrl3kRI+NOpehEBSv1WHNVLfubWaC8SIfXTtokWgnL1pJLYCAiFyf/2b/yip9bW/wPdNx45Q6cAAA==")))
open("/content/montage/logo_locapay.png", "wb").write(base64.b64decode("..."))
sys.path.insert(0, "/content/montage")
import importlib, montage_auto as M; importlib.reload(M)
import torch
print("✅ Installation terminée —", "GPU actif 🚀" if torch.cuda.is_available() else "⚠️ pas de GPU (ça marchera, mais plus lentement)")

In [ ]:
#@title 2️⃣ Réglages
#@markdown **Couleurs des écrans de couleur et de l'écran de fin**
couleur_1 = "#2E2C51" #@param {type:"string"}
couleur_2 = "#0AC2A4" #@param {type:"string"}
ecrans_de_couleur = 2 #@param {type:"slider", min:0, max:4, step:1}
#@markdown **Sous-titres et découpe**
position_sous_titres = 0.66 #@param {type:"slider", min:0.40, max:0.85, step:0.01}
couper_les_hesitations = True #@param {type:"boolean"}
pause_max_conservee = 0.45 #@param {type:"slider", min:0.20, max:1.00, step:0.05}
#@markdown **Son**
effets_sonores = "léger" #@param ["léger", "aucun"]
ajouter_une_musique = False #@param {type:"boolean"}
volume_musique = 0.18 #@param {type:"slider", min:0.05, max:0.50, step:0.01}
#@markdown **Logo** (cochez « autre_logo » pour charger un PNG à la place du logo Locapay)
logo = True #@param {type:"boolean"}
autre_logo = False #@param {type:"boolean"}
#@markdown **Transcription** — mots rares à bien reconnaître, séparés par des virgules
vocabulaire = "Locapay, démarcheurs, loyer, propriétaire, locataire, Cotonou, Bénin" #@param {type:"string"}
modele = "turbo" #@param ["turbo", "large-v3", "medium", "small"]
#@markdown **Vidéo déjà sur Google Drive ?** Indiquez son chemin (ex. `MyDrive/Videos/rush.mp4`), sinon laissez vide.
video_sur_drive = "" #@param {type:"string"}

R = M.Reglages(couleur_1=couleur_1, couleur_2=couleur_2, nb_ecrans_couleur=ecrans_de_couleur,
               position_sous_titres=position_sous_titres, supprimer_hesitations=couper_les_hesitations,
               pause_max=pause_max_conservee, effets_sonores=effets_sonores, volume_musique=volume_musique,
               logo_en_coin=logo, fichier_logo="/content/montage/logo_locapay.png" if logo else "",
               police_serif="/content/montage/polices/serif.ttf", police_grasse="/content/montage/polices/gras.ttf")
print("✅ Réglages enregistrés")

In [ ]:
#@title 3️⃣ Charger la vidéo (et la musique)
from google.colab import files
import shutil
os.makedirs("/content/montage/entree", exist_ok=True)
EXT_V = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".3gp")
if video_sur_drive.strip():
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = os.path.join("/content/drive", video_sur_drive.strip().lstrip("/"))
    assert os.path.exists(VIDEO), f"Fichier introuvable : {VIDEO}"
else:
    print("👉 Sélectionnez la vidéo brute :")
    up = files.upload()
    vids = [n for n in up if n.lower().endswith(EXT_V)]
    assert vids, "Aucune vidéo reconnue (formats : mp4, mov, m4v, avi, mkv, webm)."
    VIDEO = f"/content/montage/entree/{vids[0]}"
    shutil.move(vids[0], VIDEO)
MUSIQUE = None
if ajouter_une_musique:
    print("👉 Sélectionnez la musique (mp3, wav, m4a…) :")
    up = files.upload()
    if up:
        n = list(up)[0]; MUSIQUE = f"/content/montage/entree/{n}"; shutil.move(n, MUSIQUE)
if logo and autre_logo:
    print("👉 Sélectionnez votre logo (PNG, fond transparent de préférence) :")
    up = files.upload()
    if up:
        n = list(up)[0]; R.fichier_logo = f"/content/montage/entree/{n}"; shutil.move(n, R.fichier_logo)
DUREE = M.duree_video(VIDEO)
print(f"✅ Vidéo : {os.path.basename(VIDEO)} ({DUREE:.1f} s)" + (f" — musique : {os.path.basename(MUSIQUE)}" if MUSIQUE else ""))

In [ ]:
#@title 4️⃣ Transcription automatique
mots_bruts = M.transcrire(VIDEO, modele=modele, langue="fr", vocabulaire=vocabulaire)
print("\n📝 Transcription :\n")
print(M.texte_transcription(mots_bruts))

In [ ]:
#@title 5️⃣ Corrections (facultatif) et plan de montage
#@markdown Format : `mot_mal_reconnu=bon_mot; autre=correction` — ex. `Locapé=Locapay; démarcheur=démarcheurs`
corrections = "Locapé=Locapay; Lokapay=Locapay; Locapaï=Locapay" #@param {type:"string"}
mots = M.appliquer_corrections(mots_bruts, corrections)
PLAN = M.construire_plan(mots, DUREE, R)
print(M.resume_plan(PLAN))
print("\nLes mots en MAJUSCULES seront en gras. Relancez cette étape après une correction.")

In [ ]:
#@title 6️⃣ Montage et téléchargement
from IPython.display import Video, display
os.makedirs("/content/montage/sortie", exist_ok=True)
base = os.path.splitext(os.path.basename(VIDEO))[0]
SORTIE = f"/content/montage/sortie/{base}_monte.mp4"
print("🎬 Montage en cours…")
M.monter(VIDEO, PLAN, R, SORTIE, musique=MUSIQUE)
M.ecrire_srt(PLAN, SORTIE.replace(".mp4", ".srt"))
open(SORTIE.replace(".mp4", "_transcription.txt"), "w", encoding="utf-8").write(M.texte_transcription(mots))
print("✅ Terminé !")
display(Video(SORTIE, embed=True, width=320))
files.download(SORTIE)

---
**Fichiers produits** (dossier `/content/montage/sortie`, panneau 📁 à gauche) : la vidéo montée, les sous-titres `.srt` et la transcription `.txt`.

**Astuces**
- Filmez en vertical, la personne au centre, avec un son propre : la transcription n'en sera que meilleure.
- Trop de coupes ? Montez « pause_max_conservee » (étape 2). Pas assez ? Baissez-la.
- Sous-titres qui cachent les mains ou le visage ? Ajustez « position_sous_titres ».